# Segmento 3: Integrazione Trello

L'agente ora ha **due tool**: uno per cercare ricette, uno per aggiungere elementi alla lista della spesa su Trello.

Con una sola richiesta, l'agente dovrà:
1. Cercare la ricetta giusta
2. Estrarne gli ingredienti
3. Aggiungerli alla lista della spesa, uno per uno

Board Trello pubblica: https://trello.com/b/9DfwEVKZ/lista-della-spesa

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
import requests
import json, os

load_dotenv()
client = OpenAI()
qdrant = QdrantClient(host="localhost", port=6333)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-5.4-nano"
COLLECTION = "recipes"

# Credenziali Trello
TRELLO_API_KEY = os.getenv("TRELLO_API_KEY")
TRELLO_TOKEN = os.getenv("TRELLO_TOKEN")
TRELLO_LIST_ID = os.getenv("TRELLO_LIST_ID")

# Verifichiamo che Qdrant abbia i dati della sessione 2
if not qdrant.collection_exists(COLLECTION):
    print(f"La collection '{COLLECTION}' non esiste!")
    print("Vai nel notebook session2/segment_2.ipynb ed eseguilo per crearla.")
else:
    print(f"Qdrant: {qdrant.get_collection(COLLECTION).points_count} ricette")
    print(f"Trello: configurato ({'ok' if TRELLO_API_KEY else 'MANCANTE'})")

Qdrant: 1000 ricette
Trello: configurato (ok)


## Due tool per l'agente

1. `retrieve_recipes`: cerca ricette nel database (come prima)
2. `add_to_shopping_list`: aggiunge un elemento alla lista della spesa su Trello

In [7]:
# Schema dei tool — quello che l'LLM "vede"
tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_recipes",
            "description": "Cerca ricette nel database per similarità semantica.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "La query di ricerca per trovare ricette rilevanti"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_to_shopping_list",
            "description": "Aggiunge un elemento alla lista della spesa su Trello.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item_name": {
                        "type": "string",
                        "description": "Nome dell'ingrediente da aggiungere"
                    },
                    "quantity": {
                        "type": "string",
                        "description": "Quantità necessaria (es. '500g', '2 spicchi', '1 cucchiaio')"
                    }
                },
                "required": ["item_name"]
            }
        }
    }
]

print(f"Tool registrati: {[t['function']['name'] for t in tools]}")

Tool registrati: ['retrieve_recipes', 'add_to_shopping_list']


In [8]:
# Le funzioni vere

def retrieve_recipes(query, top_n=5):
    """Cerca le ricette più rilevanti in Qdrant."""
    response = client.embeddings.create(input=[query], model=EMBED_MODEL)
    query_vector = response.data[0].embedding

    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        limit=top_n,
        with_payload=True,
    )

    recipes = []
    for point in results.points:
        recipes.append({
            "title": point.payload["title"],
            "ingredients": point.payload["ingredients"],
            "instructions": point.payload["instructions"],
            "score": round(point.score, 4),
        })
    return recipes


def add_to_shopping_list(item_name, quantity=""):
    """Aggiunge un elemento alla lista della spesa su Trello."""
    card_name = f"{item_name} - {quantity}" if quantity else item_name

    resp = requests.post(
        "https://api.trello.com/1/cards",
        params={
            "key": TRELLO_API_KEY,
            "token": TRELLO_TOKEN,
            "idList": TRELLO_LIST_ID,
            "name": card_name,
        },
    )
    assert resp.status_code == 200, f"Errore Trello: {resp.status_code} - {resp.text}"
    return {"status": "ok", "card_name": card_name}


# Mappa nome → funzione per il loop agentico
TOOL_FUNCTIONS = {
    "retrieve_recipes": retrieve_recipes,
    "add_to_shopping_list": add_to_shopping_list,
}

print("Funzioni pronte")

Funzioni pronte


## Il loop agentico

L'agente gira in un loop:
1. Manda i messaggi all'LLM
2. Se l'LLM chiede di usare un tool → lo eseguiamo e torniamo al punto 1
3. Se l'LLM risponde con testo → abbiamo finito

In [9]:
SYSTEM_PROMPT = """Sei un assistente culinario con accesso a due strumenti:

1. retrieve_recipes: cerca ricette nel database per similarità semantica
2. add_to_shopping_list: aggiunge un elemento alla lista della spesa su Trello

Quando l'utente ti chiede di trovare una ricetta e preparare la lista della spesa:
1. Cerca prima la ricetta più adatta con retrieve_recipes
2. Scegli la ricetta migliore tra i risultati
3. Aggiungi ogni ingrediente alla lista della spesa con add_to_shopping_list, uno alla volta

Rispondi sempre in italiano."""


def run_agent(user_message):
    """Esegue l'agente con tool calling in loop."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    print(f"Utente: {user_message}\n")
    step = 1

    while True:
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=messages,
            tools=tools,
        )

        msg = response.choices[0].message
        messages.append(msg)

        # Se non ci sono tool call, abbiamo finito
        if not msg.tool_calls:
            print(f"\n--- Risposta finale ---\n\n{msg.content}")
            break

        # Eseguiamo ogni tool call
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)

            print(f"  Step {step}: {name}({json.dumps(args, ensure_ascii=False)})")

            fn = TOOL_FUNCTIONS[name]
            result = fn(**args)

            if name == "retrieve_recipes":
                print(f"         -> {len(result)} ricette trovate")
                for r in result:
                    print(f"            {r['score']:.4f}  {r['title']}")
            elif name == "add_to_shopping_list":
                print(f"         -> Aggiunto su Trello")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })
            step += 1

    return messages

print("Agente pronto")

Agente pronto


## Proviamolo

Una sola richiesta, l'agente deve cercare la ricetta e aggiungere gli ingredienti su Trello.

Tenete d'occhio la board: https://trello.com/b/9DfwEVKZ/lista-della-spesa

In [10]:
run_agent("Trova una ricetta con il pollo e metti gli ingredienti nella lista della spesa");

Utente: Trova una ricetta con il pollo e metti gli ingredienti nella lista della spesa

  Step 1: retrieve_recipes({"query": "ricetta con il pollo"})
         -> 5 ricette trovate
            0.4638  Lemon-Chicken Drumsticks
            0.4636  Skillet Roast Chicken with Fennel, Parsnips, and Scallions
            0.4557  Parmesan Chicken Cutlets
            0.4526  Chicken Paillards with Tomato, Basil, and Roasted-Corn Relish
            0.4522  Herb-Rubbed Cast-Iron Chicken with Pan Sauce
  Step 2: add_to_shopping_list({"item_name": "12 chicken drumsticks (circa 1.4 kg)", "quantity": ""})
         -> Aggiunto su Trello
  Step 3: add_to_shopping_list({"item_name": "Zeste e succo di 2 limoni", "quantity": ""})
         -> Aggiunto su Trello
  Step 4: add_to_shopping_list({"item_name": "2 cucchiai di timo fresco tritato", "quantity": ""})
         -> Aggiunto su Trello
  Step 5: add_to_shopping_list({"item_name": "2 cucchiai di olio extravergine d'oliva", "quantity": ""})
         -> Ag

## Cosa abbiamo costruito

Un **agente** che:
1. Riceve una richiesta in linguaggio naturale
2. **Decide da solo** la sequenza di azioni da compiere
3. Cerca nel database, estrae gli ingredienti, li aggiunge su Trello
4. Tutto in **una sola interazione**

Il pattern è lo stesso per qualsiasi integrazione: basta definire nuovi tool e l'agente impara ad usarli.

Torniamo alle slides per il recap finale.